# BTL02 – Chạy thực nghiệm trên Colab / Kaggle (GPU)

Phụ trách: Huỳnh Ngọc Trúc

Notebook này chỉ gọi lại đúng các lệnh chạy trên máy (`python -m src.train`, `python -m experiments.run_E*`), không chứa code mô hình riêng.

**Trước khi mở notebook:** commit và push code mới nhất lên GitHub, vì notebook `git clone` repo.

| | Colab | Kaggle |
|---|---|---|
| GPU | Runtime → Change runtime type → T4 GPU | Settings → Accelerator → GPU T4 x2 hoặc P100; bật Internet |
| Dữ liệu | Đặt `fer2013.csv` ở `MyDrive/fer2013.csv` | Add Input → dataset `nicolejyt/facialexpressionrecognition` |
| Mất kết nối | `results/` được nối vào `MyDrive/BTL02_results`, chạy lại ô là tiếp tục run còn thiếu | Dùng **Save Version → Save & Run All** để chạy nền (tối đa 12 giờ), kết quả ở tab Output |

Chạy xong: tải file `results_*.zip`, giải nén đè vào thư mục `BTL02 Emotion Recognition/` trong repo, rồi ghi nhật ký từng run (`/btl02-ghi-run <mã run>`), cột thời gian ghi kèm loại GPU.

In [ ]:
!nvidia-smi

In [ ]:
import glob, os, shutil, subprocess, time

KAGGLE = os.path.exists("/kaggle")
COLAB = not KAGGLE and os.path.exists("/content")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO = os.path.join(WORK, "deep-learning-assignments")
URL = "https://github.com/TysonNgo25/deep-learning-assignments.git"

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, REPO], check=True)
os.chdir(os.path.join(REPO, "BTL02 Emotion Recognition"))
print("Thư mục làm việc:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Dữ liệu: Kaggle lấy từ Input, Colab lấy từ Google Drive.
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/fer2013.csv"):
    found = glob.glob("/kaggle/input/**/fer2013.csv", recursive=True) +             glob.glob("/content/drive/MyDrive/fer2013.csv")
    if not found:
        raise FileNotFoundError("Không thấy fer2013.csv (Kaggle: Add Input; Colab: MyDrive/fer2013.csv)")
    shutil.copy(found[0], "data/fer2013.csv")
print("data/fer2013.csv:", os.path.getsize("data/fer2013.csv") // 2**20, "MB")

# Colab: nối results/ vào Drive để không mất log khi bị ngắt kết nối.
if COLAB and not os.path.islink("results"):
    drive_results = "/content/drive/MyDrive/BTL02_results"
    os.makedirs(drive_results, exist_ok=True)
    shutil.rmtree("results")
    os.symlink(drive_results, "results")
    print("results/ ->", drive_results)

In [ ]:
!pip install -q -r requirements.txt
!python -c "from src.data import load_fer2013, check_counts; check_counts(load_fer2013(splits=('train', 'val')))"

## 1. Chạy thử (vài giây) rồi BASE đầy đủ

Chạy thử ghi vào `results/_thu/`, không ảnh hưởng kết quả thật.

In [ ]:
!python -m src.train --run BASE --thu

In [ ]:
!python -m src.train --run BASE --change "baseline"

## 2. Các nhóm thực nghiệm

Run đã có log sẽ được bỏ qua, nên chạy lại một ô sau khi bị ngắt là tiếp tục từ run còn thiếu. Kiểm nhanh config của cả nhóm trước bằng `--thu` (mỗi run 1 epoch × 5 batch).

E2 tự lấy kiến trúc tốt nhất từ `results/E1/summary.csv`, nên phải chạy sau E1.

In [ ]:
!python -m experiments.run_E1

In [ ]:
!python -m experiments.run_E2

In [ ]:
!python -m experiments.run_E3

In [ ]:
!python -m experiments.run_E4

In [ ]:
!python -m experiments.run_E5

In [ ]:
!python -m experiments.run_E6

## 3. Đóng gói kết quả để tải về

Không kèm `checkpoints/` (nặng, chỉ cần checkpoint mô hình cuối) và `_thu/`.

In [ ]:
name = f"results_{time.strftime('%Y%m%d-%H%M')}.zip"
zip_path = os.path.join(WORK, name)
subprocess.run(["zip", "-qr", zip_path, "results/", "-x", "results/_thu/*", "*/checkpoints/*"], check=True)
print("Đã tạo", zip_path, os.path.getsize(zip_path) // 1024, "KB")
if COLAB:
    from google.colab import files
    files.download(zip_path)